# WorldSplat-Prune — TUM2TWIN retrieval

Downloads a **trainable scene** (posed imagery source) and the **reference
geometry** (the C2 label source: LoD/CityGML models + LiDAR) for the urban
benchmark. Preparation (HLOC SfM, reference rasterisation/labelling) is **not**
done here — that stays in `Prepare_Datasets.ipynb` / `worldsplat_prune.prep`,
which read from `raw/` and write to `Dataset/`.

| subset | source | lands at |
|---|---|---|
| UAS photographs (trainable scene) | Zenodo `14548134` → `Images.zip` | archive on `$RAW_DIR/tum2twin/uas_campus/UAS_Images.zip`, images extracted to local disk |
| OPF camera calibration + sparse reconstruction | Zenodo `15282970` → `opf.zip` | archive on `$RAW_DIR/tum2twin/uas_campus/opf.zip`, extracted to local disk |
| UAS LiDAR (reference point cloud) | Zenodo `14548134` → `*.laz` | `$DATASET_ROOT/tum2twin/reference/uas_lidar/` |
| CityGML LoD2/LoD3 buildings + LoD3 vegetation + merged CAD (reference geometry) | GitLab LFS `tum-gis/tum2twin-datasets` | `$DATASET_ROOT/tum2twin/reference/citygml/`, `.../cad/` |
| Bavarian ALS + LoD2 GML tiles (reference geometry) | Bavarian open data (HTTP) | `$DATASET_ROOT/tum2twin/reference/als/`, `.../lod2-bavaria/` |

Paths follow the existing convention:

```
HOME_DIR     = /content/drive/MyDrive/Magister/Thesis/Script
RAW_DIR      = $HOME_DIR/raw        # raw imagery + scratch
DATASET_ROOT = $HOME_DIR/Dataset    # prepared scenes + reference geometry
```

The trainable scene is the **UAS photograph set** (`Images.zip`, ~5.9 GB) plus the
**OPF** archive (`opf.zip`, ~1.8 GB) that carries its camera calibration and
sparse reconstruction — together these give a *posed* scene without running SfM
(`opf2colmap`, §3b). The LiDAR counterpart from the same campaign lands as
reference geometry.

**Local vs Drive.** Mounted Drive is slow for thousands of small files, while the
local runtime disk (`/content/_retrieve`) is fast but ephemeral. So each archive
is **persisted on Drive**, then by default **staged to local and extracted
there** — extraction and downstream work (conversion, SfM, labelling) happen
locally. Set `PERSIST_EXTRACT` if you also want a convenience copy of the images
on Drive (§2).

> **Not scripted here** (manual/brittle hosts): street-level videos (SharePoint),
> ZAHA / TUM-MLS-24 / TUM-TLS-24 (LRZ share links), Munich solitary vegetation
> (IOER, registration). Add them from the site if a study needs them.

Full guide: `docs/datasets.md`; urban framing: `docs/worldsplat_prune_spec.md`
(§2.3, D19), `docs/tasks/c1_c3_urban_implementation.md` (T3).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Configuration

Toggle each subset independently. Re-running is safe: a file/folder already on
Drive is skipped. `SCENE_NAME` names the trainable-scene folder that
`Prepare_Datasets.ipynb` will SfM into a `sparse/0` COLMAP model.

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

HOME_DIR     = "/content/drive/MyDrive/Magister/Thesis/Script"
RAW_DIR      = Path(HOME_DIR) / "raw"                    # raw downloads + scratch (Drive)
DATASET_ROOT = Path(HOME_DIR) / "Dataset"                # prepared scenes + reference (Drive)

# TUM2TWIN material (mirrors the mipnerf360 <root>/<scene> split)
T2T_RAW = RAW_DIR / "tum2twin"                          # raw imagery (feeds conversion / SfM)
T2T_REF = DATASET_ROOT / "tum2twin" / "reference"      # reference geometry (C2 label source)
SCENE_NAME = "uas_campus"                               # trainable-scene folder name

# --- subset toggles -------------------------------------------------------- #
FETCH_SCENE     = True    # UAS photographs -> local images/ (Zenodo 14548134)
FETCH_OPF       = True    # OPF camera calibration + sparse reconstruction (Zenodo 15282970)
FETCH_UAS_LIDAR = True    # UAS laser scans  -> reference/uas_lidar/ (Zenodo 14548134)
FETCH_CITYGML   = True    # LoD2/LoD3 buildings + LoD3 vegetation + merged CAD (Git LFS)
FETCH_ALS       = True    # Bavarian ALS laz tiles + LoD2 GML

# --- local-vs-Drive policy ------------------------------------------------- #
# Drive is slow for many small files; the local runtime disk is fast but
# ephemeral. Keep one immutable archive on Drive, work on local disk.
STAGE_LOCAL     = True    # copy the archive Drive->local before extracting (recommended)
PERSIST_EXTRACT = False   # ALSO keep extracted images on Drive (convenience; +~5.5 GiB)

SCRATCH = Path("/content/_retrieve")                    # fast local disk (not Drive)
for p in (RAW_DIR, DATASET_ROOT, T2T_RAW, T2T_REF, SCRATCH):
    p.mkdir(parents=True, exist_ok=True)
os.environ["HOME_DIR"]     = HOME_DIR
os.environ["RAW_DIR"]      = str(RAW_DIR)
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)
print("scene archive (Drive):", T2T_RAW / SCENE_NAME / "UAS_Images.zip")
print("opf   archive (Drive):", T2T_RAW / SCENE_NAME / "opf.zip")
print("scene images  (local):", SCRATCH / SCENE_NAME / "images")
print("reference           :", T2T_REF)


def _fetch(url, dst):
    """wget a URL to dst unless it already exists; returns the local path."""
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 0:
        print("skip (have):", dst.name)
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    print("downloading:", dst.name)
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(dst), url], check=True)
    return dst


def _stage(dst, src):
    """Copy a large file from Drive to the fast local disk, skipping if staged."""
    dst, src = Path(dst), Path(src)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print("skip (staged):", dst.name)
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    print("staging:", src.name, "Drive -> local")
    shutil.copy2(src, dst)
    return dst

## 2b. Size budget

Expected transfer and on-disk sizes, verified from the sources (Zenodo API,
GitLab LFS pointers, Bavarian `Content-Length`). Run this **before**
downloading to plan the budget — and again afterwards: enabled subsets show
their expected size, and whatever already exists under `RAW_DIR` / `reference`
is measured.

In [ ]:
# Expected transfer sizes in bytes, verified from the sources:
#   Zenodo API (14548134 / 15282970), GitLab LFS pointers, Bavarian Content-Length.
EXPECTED_BYTES = {
    "uas_images_zip": 5_906_891_973,   # Images.zip (persisted on Drive)
    "opf_zip":        1_936_493_976,   # OPF calibration + sparse reconstruction (Zenodo 15282970)
    "uas_lidar":      3_380_795_367,   # 3 laz + trajectory + docu
    "citygml_cad":      997_215_673,   # LoD2/LoD3 buildings + vegetation + merged CAD
    "als_laz":          470_768_749,   # 4 Bavarian ALS tiles
    "lod2_bavaria":     309_590_816,   # 2 LoD2 GML tiles
}
ENABLED = {
    "uas_images_zip": FETCH_SCENE,
    "opf_zip":        FETCH_OPF,
    "uas_lidar":      FETCH_UAS_LIDAR,
    "citygml_cad":    FETCH_CITYGML,
    "als_laz":        FETCH_ALS,
    "lod2_bavaria":   FETCH_ALS,
}
# Persistent Drive targets (the archive, plus the extracted mirror if kept).
_scene_targets = [T2T_RAW / SCENE_NAME / "UAS_Images.zip"]
if PERSIST_EXTRACT:
    _scene_targets.append(T2T_RAW / SCENE_NAME / "images")
TARGETS = {
    "uas_images_zip": tuple(_scene_targets),
    "opf_zip":        T2T_RAW / SCENE_NAME / "opf.zip",
    "uas_lidar":      T2T_REF / "uas_lidar",
    "citygml_cad":    (T2T_REF / "citygml", T2T_REF / "cad"),
    "als_laz":        T2T_REF / "als",
    "lod2_bavaria":   T2T_REF / "lod2-bavaria",
}


def _du(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return 0


def _gib(n):
    return n / 1024 ** 3


def _total(v):
    return sum(_du(p) for p in (v if isinstance(v, (list, tuple)) else (v,)))


print(f"{'subset':16s} {'enabled':>7s} {'expected':>9s} {'on disk':>9s}")
print("-" * 46)
exp_total = disk_total = 0
for key, size in EXPECTED_BYTES.items():
    if ENABLED[key]:
        exp_total += size
    disk = _total(TARGETS[key])
    disk_total += disk
    print(f"{key:16s} {str(ENABLED[key]):>7s} {_gib(size):8.2f}G {_gib(disk):8.2f}G")
print("-" * 46)
print(f"{'TOTAL':16s} {'':>7s} {_gib(exp_total):8.2f}G {_gib(disk_total):8.2f}G  (GiB)")
scratch = _gib(2 * EXPECTED_BYTES["uas_images_zip"] + 2 * EXPECTED_BYTES["opf_zip"]) + 1.0
print(f"  enabled download : {_gib(exp_total):.2f} GiB")
print(f"  persistent (Drive): {_gib(disk_total):.2f} GiB")
print(f"  transient scratch: ~{scratch:.1f} GiB peak (staged archives + extracted data + git clone)")

## 3. Trainable scene + UAS LiDAR (Zenodo `14548134`)

One Zenodo record ([UAV Laser Scanning and Photogrammetry of TUM Downtown
Campus](https://zenodo.org/records/14548134)) carries both the UAS photographs
(`Images.zip`) and the UAS LiDAR scans (reference point clouds).

The archive is **persisted on Drive** (`UAS_Images.zip`) and, with `STAGE_LOCAL`,
copied once to the local runtime disk and **extracted there** — so SfM reads
from fast local disk, not the Drive mount. Extracted images are flattened into
`images/` for `hloc_sfm`. Set `PERSIST_EXTRACT` to also mirror them to Drive.

In [ ]:
ZENODO_UAS = "https://zenodo.org/api/records/14548134/files"

if FETCH_SCENE:
    # 1. persistent archive on Drive (the single source of truth)
    drive_zip = _fetch(f"{ZENODO_UAS}/Images.zip/content",
                       T2T_RAW / SCENE_NAME / "UAS_Images.zip")
    # 2. optionally stage it to the fast local disk before extracting
    zip_path = _stage(SCRATCH / "UAS_Images.zip", drive_zip) if STAGE_LOCAL else drive_zip

    images_dir = SCRATCH / SCENE_NAME / "images"        # local working copy
    if images_dir.exists() and any(images_dir.iterdir()):
        print("skip (have images):", images_dir)
    else:
        staging = SCRATCH / "uas_images"
        staging.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(staging)
        images_dir.mkdir(parents=True, exist_ok=True)
        n = 0
        for pat in ("*.jpg", "*.JPG", "*.jpeg", "*.JPEG", "*.png", "*.PNG"):
            for f in staging.rglob(pat):
                shutil.move(str(f), images_dir / f.name)
                n += 1
        shutil.rmtree(staging, ignore_errors=True)
        print(f"{n} images -> {images_dir}  (local runtime)")

    # 3. optional convenience copy of the extracted images on Drive
    if PERSIST_EXTRACT:
        mirror = T2T_RAW / SCENE_NAME / "images"
        if mirror.exists() and any(mirror.iterdir()):
            print("skip (have Drive images):", mirror)
        else:
            shutil.copytree(images_dir, mirror)
            print("mirrored images ->", mirror, "  (Drive)")
else:
    print("scene retrieval skipped.")

if FETCH_UAS_LIDAR:
    out = T2T_REF / "uas_lidar"
    uas_keys = (
        "TUM_Downtown_ULS_20241217_nadir.laz",
        "TUM_Downtown_ULS_20241217_manual.laz",
        "TUM_Downtown_ULS_20241217_manual_5mm_fillparts.laz",
        "TUM_Downtown_ULS_trajectory_20241217.txt",
        "TUM_Downtown_ULS_Photogrammetry_20241217_docu.pdf",
    )
    for key in uas_keys:
        _fetch(f"{ZENODO_UAS}/{key}/content", out / key)
else:
    print("UAS LiDAR retrieval skipped.")

## 3b. OPF — camera calibration + sparse reconstruction (Zenodo `15282970`)

`opf.zip` is the Pix4D **Open Photogrammetry Format** archive: camera
calibration, image geolocation, and the sparse reconstruction for the UAS
photographs. Paired with the images from §3 it gives a **posed** scene with no
SfM step — convert it with `pyopf`:

```bash
pip install "pyopf[tools]"
opf2colmap <local>/uas_campus/opf/project.opf   # -> cameras.txt images.txt points3D.txt
```

`opf.zip` lives in the **latest** version of the UAS record (`15282970`, v2 of
`14548134`); the images archive is byte-identical across both, so the pair stays
consistent. Persisted on Drive, staged and extracted locally, like §3.

In [ ]:
ZENODO_UAS_OPF = "https://zenodo.org/api/records/15282970/files"   # latest v of 14548134

if FETCH_OPF:
    # 1. persistent archive on Drive
    drive_opf = _fetch(f"{ZENODO_UAS_OPF}/opf.zip/content",
                       T2T_RAW / SCENE_NAME / "opf.zip")
    # 2. optionally stage to the fast local disk before extracting
    opf_zip = _stage(SCRATCH / "opf.zip", drive_opf) if STAGE_LOCAL else drive_opf

    opf_dir = SCRATCH / SCENE_NAME / "opf"              # local working copy
    if opf_dir.exists() and any(opf_dir.rglob("*.opf")):
        print("skip (have opf):", opf_dir)
    else:
        opf_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(opf_zip) as z:
            z.extractall(opf_dir)
        hits = sorted(opf_dir.rglob("*.opf"))
        print("opf extracted ->", opf_dir)
        for h in hits[:5]:
            print("  project file:", h)
        if not hits:
            print("  WARNING: no *.opf found under", opf_dir)
else:
    print("OPF retrieval skipped.")

## 4. Reference geometry — CityGML / CAD (GitLab LFS)

The 3D model datasets live in a [Git LFS](https://git-lfs.com/) repo. Only the
reference-geometry folders are checked out (sparse) to avoid pulling the whole
repository: LoD2 + LoD3 building models, LoD3 vegetation, and the merged LoD3
city CAD (`obj/`). The clone happens on the fast local disk; only the selected
trees are copied to Drive.

In [ ]:
DATASET_REPO = "https://gitlab.lrz.de/tum-gis/tum2twin-datasets.git"
REPO_STAGE   = SCRATCH / "tum2twin-datasets"       # clone on fast disk
SPARSE = (
    "citygml/lod2-building-datasets",
    "citygml/lod3-building-datasets",
    "citygml/lod3-vegetation-datasets",
    "obj/lod3_merged_city_model",
)


def _ensure_git_lfs():
    if shutil.which("git-lfs"):
        return
    subprocess.run("sudo apt-get update -qq && sudo apt-get install -y -qq git-lfs",
                   shell=True, check=True)
    subprocess.run(["git", "lfs", "install"], check=True)


def _copy_tree(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists():
        print("skip (have):", dst)
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print("copied", src.name, "->", dst)


if FETCH_CITYGML:
    _ensure_git_lfs()
    if not (REPO_STAGE / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", DATASET_REPO, str(REPO_STAGE)], check=True)
    subprocess.run(["git", "-C", str(REPO_STAGE), "sparse-checkout", "set", *SPARSE], check=True)
    _copy_tree(REPO_STAGE / "citygml/lod2-building-datasets",  T2T_REF / "citygml/lod2-building-datasets")
    _copy_tree(REPO_STAGE / "citygml/lod3-building-datasets",  T2T_REF / "citygml/lod3-building-datasets")
    _copy_tree(REPO_STAGE / "citygml/lod3-vegetation-datasets", T2T_REF / "citygml/lod3-vegetation-datasets")
    _copy_tree(REPO_STAGE / "obj/lod3_merged_city_model",      T2T_REF / "cad/lod3_merged_city_model")
else:
    print("CityGML/CAD retrieval skipped.")

## 5. Reference geometry — Bavarian ALS + LoD2 GML (open data)

Direct HTTP from the Bavarian State Mapping Agency: the ALS `*.laz` tiles that
cover the campus and the matching LoD2 CityGML tiles (the same 1–3 cm cadastre
the TUM2TWIN LoD models derive from).

In [ ]:
BAVARIA_LASER = "https://geodaten.bayern.de/odd_data/laser/{}.laz"
BAVARIA_LOD2  = "https://download1.bayernwolke.de/a/lod2/citygml/{}.gml"
ALS_TILES  = ("690_5335", "690_5336", "691_5335", "691_5336")
LOD2_TILES = ("690_5334", "690_5336")

if FETCH_ALS:
    laz_dir = T2T_REF / "als"
    for t in ALS_TILES:
        _fetch(BAVARIA_LASER.format(t), laz_dir / f"{t}.laz")
    gml_dir = T2T_REF / "lod2-bavaria"
    for t in LOD2_TILES:
        _fetch(BAVARIA_LOD2.format(t), gml_dir / f"{t}.gml")
else:
    print("ALS / LoD2 retrieval skipped.")

## 6. Verify what landed on Drive

In [ ]:
def _count(d, pats):
    d = Path(d)
    return sum(len(list(d.rglob(p))) for p in pats)


print("trainable scene")
archive = T2T_RAW / SCENE_NAME / "UAS_Images.zip"
opf_archive = T2T_RAW / SCENE_NAME / "opf.zip"
local_img = SCRATCH / SCENE_NAME / "images"
local_opf = SCRATCH / SCENE_NAME / "opf"
drive_img = T2T_RAW / SCENE_NAME / "images"
print(f"  archive (Drive)  : size={_gib(_du(archive)):5.2f}G  "
      f"{'ok' if archive.exists() else 'MISSING'}  -> {archive}")
print(f"  opf.zip (Drive)  : size={_gib(_du(opf_archive)):5.2f}G  "
      f"{'ok' if opf_archive.exists() else 'MISSING'}  -> {opf_archive}")
if local_img.exists():
    print(f"  images  (local)  : {_count(local_img, ('*.jpg', '*.JPG', '*.jpeg', '*.png')):5d} imgs  "
          f"size={_gib(_du(local_img)):5.2f}G  -> {local_img}")
if local_opf.exists():
    hits = sorted(local_opf.rglob("*.opf"))
    print(f"  opf     (local)  : {len(hits)} project file(s)  -> {local_opf}")
    for h in hits[:3]:
        print(f"      {h}")
if drive_img.exists():
    print(f"  images  (Drive)  : {_count(drive_img, ('*.jpg', '*.JPG', '*.jpeg', '*.png')):5d} imgs  "
          f"size={_gib(_du(drive_img)):5.2f}G  -> {drive_img}")

print("\nreference geometry")
checks = {
    "LoD2 buildings (gml)": T2T_REF / "citygml/lod2-building-datasets",
    "LoD3 buildings (gml)": T2T_REF / "citygml/lod3-building-datasets",
    "LoD3 vegetation (gml)": T2T_REF / "citygml/lod3-vegetation-datasets",
    "merged CityGML CAD":   T2T_REF / "cad/lod3_merged_city_model",
    "UAS LiDAR (laz/txt)":  T2T_REF / "uas_lidar",
    "ALS (laz)":            T2T_REF / "als",
    "LoD2 Bavaria (gml)":   T2T_REF / "lod2-bavaria",
}
ref_total = 0
for label, d in checks.items():
    n = _count(d, ("*.gml", "*.laz", "*.obj", "*.txt", "*.pdf")) if d.exists() else 0
    sz = _du(d)
    ref_total += sz
    print(f"  {label:22s} files={n:5d} size={_gib(sz):5.2f}G  {'ok' if n else 'MISSING'}  -> {d}")
print(f"  {'reference total':22s}            size={_gib(ref_total):5.2f}G")

## 7. Next steps

The raw scene is **not yet trainable** in our layout — it needs an undistorted
PINHOLE COLMAP model, and the reference geometry needs to be rasterised into
per-view labels (C2). Both belong to the preparation track:

- **Scene → COLMAP (OPF route, preferred).** The OPF archive already carries the
  camera calibration + sparse reconstruction, so no SfM is needed. In the same
  session:
  ```bash
  pip install "pyopf[tools]"
  opf2colmap /content/_retrieve/uas_campus/opf/project.opf   # cameras/images/points3D .txt
  colmap model_converter --input_path <txt> --output_path <out>/sparse/0 --output_type BIN
  colmap image_undistorter --image_path /content/_retrieve/uas_campus/images       --input_path <out>/sparse/0 --output_path <out>
  ```
  Target: `$DATASET_ROOT/tum2twin/uas_campus/{images,sparse/0}` (PINHOLE).
- **Scene → COLMAP (HLOC route).** If you skip OPF, SfM the extracted images via
  `Prepare_Datasets.ipynb` / `worldsplat_prune.prep.hloc_sfm` (slower).
- **Reference → labels (C2, T3 route (a)).** Rasterise the LoD/CityGML + LiDAR
  geometry into `semantic_reference/<stem>.npy` via the planned
  `worldsplat_prune/prep/label_from_reference.py` adapter (`--label_adapter reference`).
  See `docs/tasks/c1_c3_urban_implementation.md` (T3) — this adapter is **not
  implemented yet**.
- **Train + run.** `experiments/runs.py` with the urban scene once the COLMAP
  model exists (tiling is a study decision, spec §6 / [D22]).

**Caveats.** The UAS media are filtered but check licensing before redistribution;
the TLS release notes a `0.7551 m` Z-offset (not relevant to the UAS/LoD sets here).
Local scratch is ephemeral — Colab clears it between sessions, so re-run §3/§3b to
re-stage and re-extract (the Drive archives are not re-downloaded).